## KIT719 Project 1: Developing an Information Retrieval Application Using NLTK 

### 1. Dataset Selection ###

Using the NLTK Reuters Corpus (Option 1).

In [23]:
import nltk

# Load the Reuters Corpus
from nltk.corpus import reuters

doc_ids = reuters.fileids()
raw_texts = {d: reuters.raw(d) for d in doc_ids}          # doc_id -> raw text
categories_by_doc = {d: reuters.categories(d) for d in doc_ids}  # doc_id -> category labels

print(f"Documents loaded : {len(doc_ids)}")
print(f"Categories       : {len(reuters.categories())}")
print(f"Sample document  : {doc_ids[0]}")
print(raw_texts[doc_ids[0]][:200])

Documents loaded : 10788
Categories       : 90
Sample document  : test/14826
ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT
  Mounting trade friction between the
  U.S. And Japan has raised fears among many of Asia's exporting
  nations that the row could inflict far-reachin


### 2. Text Preprocessing Module ###

Pipeline: lowercase conversion -> tokenisation -> punctuation/numeric-noise
filtering -> stopword removal -> normalisation (stemming or POS-aware lemmatisation).
Normalisation is a parameter (`none` / `stem` / `lemma`)

In [24]:
import string
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words("english"))
numeric_pattern = re.compile(r"^\d+([.,]\d+)?$")   # filters bare numbers (low retrieval value)
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def wordnet_pos(tag):
    '''Map a Penn Treebank POS tag to WordNet's POS format for accurate lemmatisation.'''
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN


def preprocess(text, normalisation="lemma", remove_stopwords=True):
    '''Full preprocessing pipeline: tokenise, filter, normalise.'''
    text = text.lower()
    tokens = word_tokenize(text)

    # information filtering: remove punctuation, numbers, and non-alphabetic tokens
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if not numeric_pattern.match(t)]
    tokens = [t for t in tokens if any(c.isalpha() for c in t)]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    if normalisation == "stem":
        tokens = [porter.stem(t) for t in tokens]
    elif normalisation == "lemma":
        tagged = nltk.pos_tag(tokens)
        tokens = [lemmatizer.lemmatize(t, wordnet_pos(tag)) for t, tag in tagged]

    return tokens


# Three normalisation strategies on the same document
print("lemma:", preprocess(raw_texts[doc_ids[0]], normalisation="lemma")[:15])
print("stem: ", preprocess(raw_texts[doc_ids[0]], normalisation="stem")[:15])
print("none: ", preprocess(raw_texts[doc_ids[0]], normalisation="none")[:15])

lemma: ['asian', 'exporter', 'fear', 'damage', 'u.s.-japan', 'rift', 'mount', 'trade', 'friction', 'u.s.', 'japan', 'raise', 'fear', 'among', 'many']
stem:  ['asian', 'export', 'fear', 'damag', 'u.s.-japan', 'rift', 'mount', 'trade', 'friction', 'u.s.', 'japan', 'rais', 'fear', 'among', 'mani']
none:  ['asian', 'exporters', 'fear', 'damage', 'u.s.-japan', 'rift', 'mounting', 'trade', 'friction', 'u.s.', 'japan', 'raised', 'fears', 'among', 'many']


### 3. Document Indexing ###

Two structures built from the same vocabulary:
- **Inverted index** — `term -> {doc_id: term_frequency}`, giving O(1) lookup
  of which documents contain a query term.
- **TF-IDF sparse matrix** — used by the cosine-similarity ranker for fast
  vectorised scoring.

In [25]:
from collections import defaultdict, Counter

tokenised_docs = [preprocess(raw_texts[d], normalisation="lemma") for d in doc_ids]


inverted_index = defaultdict(dict) # term -> {doc_id: term_frequency}
doc_len = {}   # doc_id -> number of tokens

for doc_id, tokens in zip(doc_ids, tokenised_docs):
    doc_len[doc_id] = len(tokens)
    tf = Counter(tokens)
    for term, freq in tf.items():
        inverted_index[term][doc_id] = freq

vocab = sorted(inverted_index.keys())
print(f"Vocabulary size: {len(vocab)} unique terms")
print(f"'grain' appears in {len(inverted_index['grain'])} documents")

Vocabulary size: 29866 unique terms
'grain' appears in 301 documents


In [26]:
def candidate_docs(query_terms, inverted_index):
    candidates = set()
    for term in query_terms:
        candidates.update(inverted_index.get(term, {}).keys())
    return candidates


sample_query = preprocess("grain trade policy", normalisation="lemma")
sample_candidates = candidate_docs(sample_query, inverted_index)
print(f"Query terms: {sample_query}")
print(f"Candidate documents: {len(sample_candidates)} (out of {len(doc_ids)} total)")

Query terms: ['grain', 'trade', 'policy']
Candidate documents: 1769 (out of 10788 total)


In [27]:
import numpy as np
from scipy import sparse

term_to_col = {t: i for i, t in enumerate(vocab)}
doc_id_to_row = {d: i for i, d in enumerate(doc_ids)}
N = len(doc_ids)

# smoothed inverse document frequency per term
idf = {}
for term in vocab:
    df = len(inverted_index[term])
    idf[term] = np.log((1 + N) / (1 + df)) + 1

# build the sparse TF-IDF matrix
rows, cols, data = [], [], []
for term in vocab:
    col = term_to_col[term]
    for doc_id, tf in inverted_index[term].items():
        rows.append(doc_id_to_row[doc_id])
        cols.append(col)
        data.append((1 + np.log(tf)) * idf[term])

tfidf_matrix = sparse.csr_matrix((data, (rows, cols)), shape=(N, len(vocab)))
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (10788, 29866)


### 4. Retrieval and Ranking ###

Two independent ranking methods:
- **TF-IDF + cosine similarity** — vector space model.
- **Okapi BM25** — probabilistic ranking function with document-length
  normalisation and term-frequency saturation (`k1 = 1.5`, `b = 0.75`).

In [28]:
from sklearn.preprocessing import normalize


def query_to_tfidf_vector(query_terms):
    """Project a preprocessed query onto the document TF-IDF vector space."""
    tf = Counter(query_terms)
    cols, data = [], []
    for term, freq in tf.items():
        if term in term_to_col:
            data.append((1 + np.log(freq)) * idf[term])
            cols.append(term_to_col[term])
    return sparse.csr_matrix((data, ([0] * len(cols), cols)), shape=(1, len(vocab)))


def rank_tfidf(query_terms, top_k=10):
    """Rank candidate documents by TF-IDF cosine similarity."""
    candidates = candidate_docs(query_terms, inverted_index)
    if not candidates:
        return []
    q_vec = normalize(query_to_tfidf_vector(query_terms))
    rows = [doc_id_to_row[d] for d in candidates]
    sub_matrix = normalize(tfidf_matrix[rows])
    scores = sub_matrix.dot(q_vec.T).toarray().ravel()
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(d, s) for d, s in ranked if s > 0][:top_k]

In [29]:
import math


def rank_bm25(query_terms, top_k=10, k1=1.5, b=0.75):
    candidates = candidate_docs(query_terms, inverted_index)
    if not candidates:
        return []

    avg_len = sum(doc_len.values()) / N
    scores = defaultdict(float)

    for term in set(query_terms):
        postings = inverted_index.get(term, {})
        if not postings:
            continue
        df = len(postings)
        term_idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
        for doc_id, tf in postings.items():
            if doc_id not in candidates:
                continue
            dl = doc_len[doc_id]
            denom = tf + k1 * (1 - b + b * dl / avg_len)
            scores[doc_id] += term_idf * (tf * (k1 + 1)) / denom

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

In [30]:
# compare both ranking methods on the same query
query_terms = preprocess("grain trade policy", normalisation="lemma")
results_tfidf = rank_tfidf(query_terms, top_k=5)
results_bm25 = rank_bm25(query_terms, top_k=5)

print("--- TF-IDF + Cosine Similarity ---")
for doc_id, score in results_tfidf:
    print(f"{doc_id:<15} {score:.4f}  {raw_texts[doc_id][:60].strip()}")

print("\n--- BM25 ---")
for doc_id, score in results_bm25:
    print(f"{doc_id:<15} {score:.4f}  {raw_texts[doc_id][:60].strip()}")

--- TF-IDF + Cosine Similarity ---
training/12558  0.3316  YEUTTER SAYS U.S. SHOULD STRESS TRADE NEGOTIATIONS AS LONG-T
training/2223   0.2771  WORLD GRAIN TRADE RECOVERY MAY BE UNDERWAY
  World grain tra
training/3311   0.2474  USDA ESTIMATES 1986/87 USSR GRAIN IMPORTS  26.0 MLN TONNES V
test/15842      0.2376  USDA ESTIMATES 1986/87 USSR GRAIN IMPORTS  28.0 MLN TONNES V
test/15844      0.2345  USDA ESTIMATES 1986 SOVIET GRAIN CROP  AT 210 MLN TONNES VS

--- BM25 ---
training/2223   13.8210  WORLD GRAIN TRADE RECOVERY MAY BE UNDERWAY
  World grain tra
training/5408   12.1866  EEP SHOULD BE USED TACTICALLY, YEUTTER SAYS
  U.S. Trade Rep
training/12558  10.4042  YEUTTER SAYS U.S. SHOULD STRESS TRADE NEGOTIATIONS AS LONG-T
training/11769  10.2173  CHINA OFFICIAL CONDEMNS GOVERNMENT GRAIN POLICY
  The grain
test/15710      10.0241  GRAIN SHIPS WAITING AT NEW ORLEANS
  Ten grain ships were lo


### 5. Query Processing Module ###

Pipeline: receive user input -> preprocess -> [spelling correction] ->
[query expansion] -> retrieve candidates -> rank -> display results.

Two query-processing techniques are supported:
- **Spelling correction** — replaces any out-of-vocabulary query term with the closest vocabulary term by Levenshtein edit distance.
- **Query expansion** — adds WordNet synonyms (already present in the vocabulary) to improve recall for differently-worded queries.

In [31]:
def correct_spelling(query_terms, vocab, max_edit_distance=2):
    vocab_set = set(vocab)
    corrected = []
    for term in query_terms:
        if term in vocab_set:          # leave genuine vocabulary words unchanged
            corrected.append(term)
            continue
        best, best_dist = None, max_edit_distance + 1
        for v in vocab_set:
            if abs(len(v) - len(term)) > max_edit_distance:
                continue
            d = nltk.edit_distance(term, v)
            if d < best_dist:
                best, best_dist = v, d
        corrected.append(best if best is not None else term)
    return corrected


def expand_query(query_terms, vocab, max_synonyms=2):
    vocab_set = set(vocab)
    expanded = list(query_terms)
    for term in query_terms:
        synonyms = set()
        for syn in wordnet.synsets(term):
            for lemma in syn.lemma_names():
                lemma = lemma.lower().replace("_", " ")
                if lemma != term and lemma in vocab_set:
                    synonyms.add(lemma)
                if len(synonyms) >= max_synonyms:
                    break
            if len(synonyms) >= max_synonyms:
                break
        expanded.extend(synonyms)
    return expanded

print("Spelling correction demo:")
print(" before:", preprocess("grian expotrs", normalisation="lemma"))
print(" after :", correct_spelling(preprocess("grian expotrs", normalisation="lemma"), vocab))

Spelling correction demo:
 before: ['grian', 'expotrs']
 after : ['graan', 'export']


In [32]:
def tfidf_term_contributions(doc_id, query_terms):
    q_vec = normalize(query_to_tfidf_vector(query_terms))
    doc_row = doc_id_to_row[doc_id]
    d_vec = normalize(tfidf_matrix[doc_row])

    contributions = []

    for term in set(query_terms):
        if term not in term_to_col:
            continue

        col = term_to_col[term]

        q_weight = q_vec[0, col]
        d_weight = d_vec[0, col]

        contribution = float(q_weight * d_weight)

        if contribution > 0:
            contributions.append((term, contribution))

    return sorted(
        contributions,
        key=lambda x: x[1],
        reverse=True
    )


# =========================================================
# BM25 term contribution
# =========================================================

def bm25_term_contributions(doc_id, query_terms, k1=1.5, b=0.75):
    contributions = []

    avg_len = sum(doc_len.values()) / N
    dl = doc_len[doc_id]

    for term in set(query_terms):

        postings = inverted_index.get(term, {})

        if doc_id not in postings:
            continue

        tf = postings[doc_id]
        df = len(postings)

        term_idf = math.log(
            1 + (N - df + 0.5) / (df + 0.5)
        )

        denom = tf + k1 * (
            1 - b + b * dl / avg_len
        )

        score = (
            term_idf
            * (tf * (k1 + 1))
            / denom
        )

        contributions.append(
            (term, score, tf)
        )

    return sorted(
        contributions,
        key=lambda x: x[1],
        reverse=True
    )


# =========================================================
# Detailed result display
# =========================================================

def display_detailed_results(
    ranked,
    raw_texts,
    query_terms,
    method
):

    if not ranked:
        print("No relevant documents found.")
        return

    for i, (doc_id, score) in enumerate(ranked, start=1):

        text = raw_texts[doc_id].strip()

        lines = text.split("\n")
        title = lines[0].strip()

        # Short preview of document
        preview = " ".join(text.split())[:250]

        # Query terms that actually appear in this document
        matched_terms = [
            term
            for term in query_terms
            if doc_id in inverted_index.get(term, {})
        ]

        # Reuters categories
        categories = categories_by_doc.get(doc_id, [])

        print("\n" + "-" * 70)
        print(f"[{i}] {title}")

        print(f"Document ID : {doc_id}")

        print(
            "Categories  :",
            ", ".join(categories)
            if categories
            else "None"
        )

        if method == "tfidf":

            print(f"Similarity  : {score:.4f}")

            contributions = tfidf_term_contributions(
                doc_id,
                query_terms
            )

            print(
                "Matched     :",
                ", ".join(matched_terms)
                if matched_terms
                else "None"
            )

            print("\nTerm contribution:")

            for term, value in contributions:
                print(
                    f"  {term:<15} {value:.4f}"
                )

        else:

            print(f"BM25 Score  : {score:.4f}")

            contributions = bm25_term_contributions(
                doc_id,
                query_terms
            )

            print(
                "Matched     :",
                ", ".join(matched_terms)
                if matched_terms
                else "None"
            )

            print("\nScore contribution:")

            for term, value, tf in contributions:
                print(
                    f"  {term:<15} "
                    f"+{value:.4f} "
                    f"(TF={tf})"
                )

        print("\nPreview:")
        print(preview + "...")


***Console Application***


**Try to run the application at this part**



In [ ]:
def run_console_app():

    print("=" * 70)
    print("KIT719 Information Retrieval System")
    print("=" * 70)

    print("Ranking methods:")
    print("  1. TF-IDF + Cosine Similarity")
    print("  2. Okapi BM25")
    print()

    print("Type a query, or 'exit' to quit.")

    while True:

        print("\n" + "=" * 70)

        query = input("User Query: ").strip()

        if query.lower() in ("exit", "quit"):
            print("Goodbye.")
            break

        if not query:
            continue


        # -------------------------------------------------
        # Query preprocessing
        # -------------------------------------------------

        original_terms = preprocess(
            query,
            normalisation="lemma"
        )

        corrected_terms = correct_spelling(
            original_terms,
            vocab
        )

        print("\nOriginal Query:")
        print(query)

        print("\nPreprocessed Query:")
        print(", ".join(original_terms))

        if corrected_terms != original_terms:

            print("\nAfter Spelling Correction:")
            print(", ".join(corrected_terms))

        terms = corrected_terms


        # -------------------------------------------------
        # TF-IDF
        # -------------------------------------------------

        tfidf_results = rank_tfidf(
            terms,
            top_k=5
        )

        print("\n")
        print("=" * 70)
        print("TF-IDF + COSINE SIMILARITY")
        print("=" * 70)

        display_detailed_results(
            tfidf_results,
            raw_texts,
            terms,
            method="tfidf"
        )


        # -------------------------------------------------
        # BM25
        # -------------------------------------------------

        bm25_results = rank_bm25(
            terms,
            top_k=5
        )

        print("\n")
        print("=" * 70)
        print("OKAPI BM25")
        print("=" * 70)

        display_detailed_results(
            bm25_results,
            raw_texts,
            terms,
            method="bm25"
        )


run_console_app()

KIT719 Information Retrieval System
Ranking methods:
  1. TF-IDF + Cosine Similarity
  2. Okapi BM25

Type a query, or 'exit' to quit.



User Query:  I was walking on Queens St this moring and I saw a War memorial



Original Query:
I was walking on Queens St this moring and I saw a War memorial

Preprocessed Query:
walk, queen, st, moring, saw, war, memorial

After Spelling Correction:
walk, queen, st, morning, saw, war, memorial


TF-IDF + COSINE SIMILARITY

----------------------------------------------------------------------
[1] INTERNATIONAL DAIRY QUEEN &lt;INDQA> 1ST QTR NET
Document ID : training/11666
Categories  : earn
Similarity  : 0.2503
Matched     : queen

Term contribution:
  queen           0.2503

Preview:
INTERNATIONAL DAIRY QUEEN &lt;INDQA> 1ST QTR NET Ended February 28. Shr 18 cts vs 13 cts Net 1,706,601 vs 1,226,609 Rev 42.7 mln vs 36.3 mln Avg shares 9,695,444 vs 9,537,043 NOTE: Company's full name is International Dairy Queen Inc....

----------------------------------------------------------------------
[2] Bank of Japan bought 200 to 300 mln dlrs this morning, dealers said.
Document ID : training/9118
Categories  : dlr, money-fx
Similarity  : 0.1439
Matched     : morning



### 6. Performance Evaluation ###

**Metrics:** Precision@10 and Recall@10 (top-of-ranking quality), F1@10
(balance of the two), and Mean Average Precision — MAP (rewards relevant
documents appearing early in the ranking, not just present in the top 10).

**Experiment:** all combinations of 3 preprocessing strategies (none / stem /
lemma) x 2 ranking methods (TF-IDF / BM25) are compared — 6 configurations
in total.

In [ ]:
docs_by_category = defaultdict(set)
for d, cats in categories_by_doc.items():
    for c in cats:
        docs_by_category[c].add(d)


def precision_recall_f1_at_k(ranked_doc_ids, relevant_set, k=10):
    top_k = ranked_doc_ids[:k]
    if not top_k:
        return 0.0, 0.0, 0.0
    hits = sum(1 for d in top_k if d in relevant_set)
    precision = hits / len(top_k)
    recall = hits / len(relevant_set) if relevant_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def average_precision(ranked_doc_ids, relevant_set, k=50):
    top_k = ranked_doc_ids[:k]
    if not relevant_set:
        return 0.0
    hits, ap_sum = 0, 0.0
    for i, d in enumerate(top_k, start=1):
        if d in relevant_set:
            hits += 1
            ap_sum += hits / i
    return ap_sum / min(len(relevant_set), k) if hits else 0.0


print("Evaluation functions defined.")

In [ ]:
TEST_CATEGORIES = ["earn", "acq", "money-fx", "grain", "crude", "trade", "interest",
                    "ship", "wheat", "corn", "money-supply", "oilseed", "sugar", "coffee"]

results = []

for norm in ("none", "stem", "lemma"):
    # rebuild the index for each normalisation strategy being compared
    tokenised = [preprocess(raw_texts[d], normalisation=norm) for d in doc_ids]
    idx_inverted = defaultdict(dict)
    idx_doc_len = {}
    for doc_id, tokens in zip(doc_ids, tokenised):
        idx_doc_len[doc_id] = len(tokens)
        for term, freq in Counter(tokens).items():
            idx_inverted[term][doc_id] = freq
    avg_len = sum(idx_doc_len.values()) / N

    for ranker_name in ("tfidf", "bm25"):
        p_list, r_list, f1_list, ap_list = [], [], [], []

        for cat in TEST_CATEGORIES:
            relevant = docs_by_category.get(cat, set())
            if not relevant:
                continue

            terms = preprocess(cat.replace("-", " "), normalisation=norm)
            candidates = set()
            for t in terms:
                candidates.update(idx_inverted.get(t, {}).keys())
            if not candidates:
                continue

            scores = defaultdict(float)
            if ranker_name == "bm25":
                for term in set(terms):
                    postings = idx_inverted.get(term, {})
                    if not postings:
                        continue
                    df = len(postings)
                    term_idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
                    for doc_id, tf in postings.items():
                        if doc_id not in candidates:
                            continue
                        dl = idx_doc_len[doc_id]
                        denom = tf + 1.5 * (1 - 0.75 + 0.75 * dl / avg_len)
                        scores[doc_id] += term_idf * (tf * 2.5) / denom
            else:
                for term in set(terms):
                    postings = idx_inverted.get(term, {})
                    df = len(postings)
                    term_idf = np.log((1 + N) / (1 + df)) + 1
                    for doc_id, tf in postings.items():
                        if doc_id not in candidates:
                            continue
                        scores[doc_id] += (1 + np.log(tf)) * term_idf

            ranked_ids = [d for d, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:50]]
            p, r, f1 = precision_recall_f1_at_k(ranked_ids, relevant, k=10)
            ap = average_precision(ranked_ids, relevant, k=50)
            p_list.append(p); r_list.append(r); f1_list.append(f1); ap_list.append(ap)

        n = len(p_list) or 1
        results.append({
            "normalisation": norm, "ranker": ranker_name,
            "precision@10": sum(p_list) / n, "recall@10": sum(r_list) / n,
            "f1@10": sum(f1_list) / n, "MAP": sum(ap_list) / n,
        })

print("Evaluation complete across 6 configurations.")

In [ ]:
print(f"{'Preprocessing':<15}{'Ranker':<10}{'Precision@10':<14}{'Recall@10':<12}{'F1@10':<10}{'MAP':<10}")
for r in results:
    print(f"{r['normalisation']:<15}{r['ranker']:<10}{r['precision@10']:<14.3f}"
          f"{r['recall@10']:<12.3f}{r['f1@10']:<10.3f}{r['MAP']:<10.3f}")

In [ ]:
import matplotlib.pyplot as plt

configs = [f"{r['normalisation']}+{r['ranker']}" for r in results]
p_scores = [r['precision@10'] for r in results]
map_scores = [r['MAP'] for r in results]

x = np.arange(len(configs))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, p_scores, width, label="Precision@10")
ax.bar(x + width / 2, map_scores, width, label="MAP")
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Retrieval performance by preprocessing + ranking configuration")
ax.legend()
plt.tight_layout()
plt.show()